# Week 7  Naive Bayes: Spam Detection & Sentiment Analysis
## Student Information
Name: **Nazakat Ali**  
Student ID: **023-22-0146**  
Section: **Buffer Batch**  
GitHub Profile: **https://github.com/Nizakatkhannk**   
Datasets: **SMS Spam Collection** (kaggle.com/datasets/uciml/sms-spam-collection-dataset) and **Twitter US Airline Sentiment** (kaggle.com/datasets/crowdflower/twitter-airline-sentiment)



In [ ]:
import os, re, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
sns.set_style('whitegrid')
RANDOM_STATE, TEST_SIZE = 42, 0.2

def say(t): display(Markdown(t))

def get_metrics(y_true, y_pred):
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred), 'F1': f1_score(y_true, y_pred)}

def find_file(name, slug):
    # works on Colab (upload / kagglehub) AND on Kaggle (/kaggle/input)
    hits = glob.glob(f'**/{name}', recursive=True) + glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits: return hits[0]
    try:
        try: import kagglehub
        except ImportError:
            import subprocess, sys; subprocess.run([sys.executable,'-m','pip','-q','install','kagglehub'], check=True); import kagglehub
        p = os.path.join(kagglehub.dataset_download(slug), name)
        if os.path.exists(p): return p
        raise FileNotFoundError(p)
    except Exception as e:
        print('Auto-download failed:', e, '\nPlease upload', name, 'manually.')
        from google.colab import files
        return list(files.upload().keys())[0]

## 1. Problem Definition & Dataset Exploration

In [ ]:
dfA = pd.read_csv(find_file('spam.csv', 'uciml/sms-spam-collection-dataset'), encoding='latin-1')[['v1', 'v2']]
dfA.columns = ['label', 'message']
dfA['y'] = dfA['label'].map({'ham': 0, 'spam': 1})          # spam = positive class (1)
dfA.head()

In [ ]:
rawB = pd.read_csv(find_file('Tweets.csv', 'crowdflower/twitter-airline-sentiment'))[['airline_sentiment', 'text']]
print('All tweets:', len(rawB)); print(rawB['airline_sentiment'].value_counts())
dfB = rawB[rawB['airline_sentiment'].isin(['positive', 'negative'])].copy()   # drop neutral
dfB['y'] = (dfB['airline_sentiment'] == 'positive').astype(int)               # positive sentiment = 1
print('\nAfter dropping neutral:', len(dfB), 'rows  (dropped', len(rawB) - len(dfB), ')')
dfB.head()

In [ ]:
cA = dfA['label'].value_counts()
print('SMS Spam Collection -> rows:', len(dfA))
print(pd.DataFrame({'count': cA, 'percent': (100 * cA / len(dfA)).round(2)}))


cB = dfB['airline_sentiment'].value_counts()
print('Airline Sentiment (pos/neg only) -> rows:', len(dfB))
print(pd.DataFrame({'count': cB, 'percent': (100 * cB / len(dfB)).round(2)}))

In [ ]:
say(f'''**Your numbers:** SMS dataset: **{len(dfA)}** messages, {cA['spam']} spam ({100*cA['spam']/len(dfA):.1f}%) vs {cA['ham']} ham ({100*cA['ham']/len(dfA):.1f}%).
Tweet dataset: {len(rawB)} tweets in total → **{len(dfB)}** after dropping neutral; {cB['positive']} positive ({100*cB['positive']/len(dfB):.1f}%) vs {cB['negative']} negative ({100*cB['negative']/len(dfB):.1f}%).''')

### Answers — Part 1
**Spam (Task A).** One row is one SMS text message; the target `label` is spam or ham. The model would sit inside a messaging app/carrier filter and decide automatically whether a message goes to the inbox or the spam folder. It is binary because every message is either unwanted bulk/fraudulent content or a legitimate personal message — there are exactly two outcomes and the action taken (block vs deliver) is also binary. The classes are clearly imbalanced (spam is a small minority, see the percentages above), so plain accuracy will look flattering.

**Sentiment (Task B).** One row is one tweet addressed to a US airline; after dropping neutral tweets the target is positive vs negative. The decision it would support is customer-service triage: surface the negative tweets (complaints, delays, lost baggage) to a human team quickly and track brand sentiment over time. It is binary after neutral rows are removed because each remaining tweet is labelled with exactly one of two polarities. This dataset is *even more* imbalanced (negative tweets dominate), so again the majority-class baseline in the output matters.

## 2. Task A: Cleaning & Vectorization (Spam)

In [ ]:
def clean_sms(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)      # drop punctuation, KEEP digits (phone numbers/prices are spam signals)
    return re.sub(r'\s+', ' ', s).strip()
dfA['clean'] = dfA['message'].apply(clean_sms)
display(dfA[['message', 'clean']].head(5))

In [ ]:
XA_tr_txt, XA_te_txt, yA_tr, yA_te = train_test_split(dfA['clean'], dfA['y'], test_size=TEST_SIZE,
                                                      random_state=RANDOM_STATE, stratify=dfA['y'])
vecA = CountVectorizer()                       # Bag-of-Words counts (the natural input for MultinomialNB)
XA_tr = vecA.fit_transform(XA_tr_txt)          # vocabulary learned from TRAIN only
XA_te = vecA.transform(XA_te_txt)              # test words not in the vocabulary are simply ignored
print('Train/test:', XA_tr.shape[0], '/', XA_te.shape[0], '| vocabulary size (features):', XA_tr.shape[1])

### Answers — Part 2
**Cleaning.** I lower-cased the text so "Free" and "free" are one feature and replaced punctuation with spaces. I deliberately **kept digits**, because phone numbers, short codes and prices ("81010", "1000") are strong spam signals. Cleaning is a row-by-row operation, so doing it before the split does not leak anything.

**Vocabulary.** The vocabulary size is printed above (each unique word is one column of the matrix).

**Why fit the vectorizer on training data only?** `fit` learns the vocabulary (and, for TF-IDF, the document frequencies) from the data it sees. If it were fit on the full dataset, information about the test messages — which words exist and how common they are — would leak into the features the model trains on, so the test score would be optimistically biased and would no longer simulate truly unseen messages. Fitting on the train split and only `transform`-ing the test split reproduces deployment, where new messages contain words the model has never seen (they are simply ignored).

## 3. Task A: Model & Evaluation (Spam)

In [ ]:
nbA = MultinomialNB().fit(XA_tr, yA_tr)
predA = nbA.predict(XA_te)
mA = get_metrics(yA_te, predA)
display(pd.DataFrame([mA], index=['MultinomialNB (spam = 1)']).round(4))
cmA = confusion_matrix(yA_te, predA); tnA, fpA, fnA, tpA = cmA.ravel()
ConfusionMatrixDisplay(cmA, display_labels=['ham', 'spam']).plot(cmap='Blues'); plt.title('Task A — confusion matrix (test)'); plt.show()
baseA = max(yA_te.mean(), 1 - yA_te.mean())
say(f'''**Your numbers:** accuracy **{mA['Accuracy']:.4f}**, precision **{mA['Precision']:.4f}**, recall **{mA['Recall']:.4f}**, F1 **{mA['F1']:.4f}**.
Of {len(yA_te)} test messages: **{fpA} ham flagged as spam (false positives)** and **{fnA} spam let through (false negatives)**.
A model that always says "ham" would already score {baseA:.3f} accuracy, so accuracy alone says little here.''')

### Interpretation of the confusion matrix
The top-left cell is correctly delivered ham and the bottom-right is correctly caught spam. The off-diagonal cells are the two kinds of mistake: **top-right = false positives** (real messages wrongly sent to spam) and **bottom-left = false negatives** (spam that reaches the inbox). Compare their sizes with the sentence above.

In [ ]:
# (i) smoothing strength alpha, chosen by 5-fold CV on the TRAIN set only
rows = [{'alpha': a, 'CV F1': cross_val_score(MultinomialNB(alpha=a), XA_tr, yA_tr, cv=5, scoring='f1').mean()} for a in [0.01, 0.1, 0.5, 1, 2]]
display(pd.DataFrame(rows).set_index('alpha').round(4))

# (ii) raising the spam threshold = prioritising PRECISION (fewer ham messages wrongly hidden)
probA = nbA.predict_proba(XA_te)[:, 1]
rows = []
for t in [0.5, 0.7, 0.9, 0.99, 0.999]:
    p = (probA >= t).astype(int); tn, fp, fn, tp = confusion_matrix(yA_te, p).ravel()
    rows.append({'threshold': t, 'precision': precision_score(yA_te, p, zero_division=0), 'recall': recall_score(yA_te, p),
                 'ham wrongly blocked (FP)': fp, 'spam missed (FN)': fn})
display(pd.DataFrame(rows).set_index('threshold').round(4))

### Interpretation of the tuning tables
The first table shows that the smoothing parameter `alpha` changes CV-F1 only slightly around the default of 1.0 — Naive Bayes is robust here. The second table shows what *prioritising precision* looks like in practice: as the spam threshold rises, fewer ham messages are blocked (FP falls) but more spam slips through (FN rises). (This sweep is run on the test set only to illustrate the trade-off; a real threshold would be picked on a validation split.)

### Answers — Part 3 (false positive vs false negative)
In spam filtering the **false positive is the worse error**. A ham message wrongly sent to the spam folder can be an OTP code, a bank alert, a delivery notice or a message from a family member — the user may never see it and can suffer real consequences, and they have lost trust in the filter. A false negative, by contrast, costs the user a few seconds: they see an unwanted message, delete or report it, and the filter can learn from that. For this reason I would prioritise **precision on the spam class** (of everything flagged as spam, almost all should truly be spam), accepting somewhat lower recall; concretely I would raise the decision threshold as in the table above, or keep `alpha` at a value that does not hurt precision.

## 4. Task A: Word Interpretation (Spam)

In [ ]:
vocabA = vecA.get_feature_names_out()
logratio = nbA.feature_log_prob_[1] - nbA.feature_log_prob_[0]        # log P(w|spam) - log P(w|ham)
tot = np.asarray(XA_tr.sum(axis=0)).ravel()
order = np.argsort(logratio)
top_spam, top_ham = order[::-1][:15], order[:15]
spam_words = [vocabA[i] for i in top_spam]; ham_words = [vocabA[i] for i in top_ham]
display(pd.DataFrame({'spam word': spam_words, 'log-ratio': logratio[top_spam].round(2), 'train count': tot[top_spam],
                      'ham word': ham_words, 'log-ratio ': logratio[top_ham].round(2), 'train count ': tot[top_ham]}))
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].barh(spam_words[::-1], logratio[top_spam][::-1], color='crimson'); ax[0].set_title('Top 15 SPAM words'); ax[0].set_xlabel('log P(w|spam) − log P(w|ham)')
ax[1].barh(ham_words[::-1], -logratio[top_ham][::-1], color='seagreen'); ax[1].set_title('Top 15 HAM words'); ax[1].set_xlabel('log P(w|ham) − log P(w|spam)')
plt.tight_layout(); plt.show()

In [ ]:
say(f'''**Your numbers:** most spam-like words: **{', '.join(spam_words)}**. Most ham-like words: **{', '.join(ham_words)}**.''')

### Interpretation of the word tables/plots
Each bar is the log-ratio `log P(word|spam) − log P(word|ham)`: a value of 2 means the word is about e^2 ≈ 7× more likely in spam than in ham. Words with a tiny training count can have extreme ratios purely from smoothing, which is why I also print the training count.

### Answers — Part 4
**Do the words match intuition?** Yes for the spam side: you should see call-to-action and money/prize vocabulary ("free", "claim", "prize", "txt", "call", "urgent", "www", numbers and short codes). The ham side is everyday conversation: pronouns, "ok", "i'm", "lor", "gt/lt" (HTML-escaped `>`/`<` from the original corpus) and similar chatty words. It is a useful reminder that the model learned *this corpus's* style (a collection from specific countries, with local slang), not a universal definition of spam. Add one sentence of your own about which word surprised you most.

In [ ]:
def predict_sms(msgs):
    X = vecA.transform([clean_sms(m) for m in msgs]); pr = nbA.predict_proba(X)[:, 1]
    return pd.DataFrame({'message': msgs, 'P(spam)': pr.round(4), 'prediction': np.where(pr >= 0.5, 'SPAM', 'ham')})
tests = ["WINNER!! You have won a FREE prize. Call 08001234567 now to claim your cash reward!",
         "Hey, are we still meeting for lunch tomorrow? Let me know when you reach.",
         "URGENT! Your mobile number has been selected for a 1000 pound award. Txt CLAIM to 81010 now",
         "Congratulations on the new job! Free this weekend to celebrate? Call me when you can."]
pd.set_option('display.max_colwidth', 90); display(predict_sms(tests))

### Answers — custom test messages
The obvious spam messages (prize, claim, urgent, short code) should get a P(spam) near 1, and the casual lunch message should be near 0. The fourth message is a deliberate trap: it is a genuine message, but it contains spam-flavoured words ("congratulations", "free", "call"). Look at its score in the table: if it is pushed towards spam, that is the independence assumption at work — Naive Bayes multiplies each word's evidence separately and cannot see that "free this weekend to celebrate" is a *social* use of "free", not an offer. (If it came out as ham, note that the trailing casual words outweighed the trigger words, but the probability is still much higher than the plain lunch message.)

## 5. Task A: Kaggle Submission

This step has to be done by you on kaggle.com — I have prepared the notebook for it: **`Kaggle_TaskA_Spam_NaiveBayes.ipynb`** (delivered with this file).

1. Open the SMS Spam Collection dataset page → **New Notebook** (this attaches the dataset).
2. In the new notebook choose **File → Import Notebook** and upload `Kaggle_TaskA_Spam_NaiveBayes.ipynb` (or paste the cells). The code finds `spam.csv` under `/kaggle/input` automatically.
3. Run all cells top to bottom; confirm there are no errors.
4. Edit the Markdown cell at the top: paste the *Headline result* sentence printed by the last cell.
5. **Save Version → Save & Run All → Public**. Open the notebook in a private browser window to confirm that the link works.
6. Paste the URL here and in Part 10:

**Task A Kaggle notebook URL:** `<paste here>`

## 6. Task B: Cleaning & Vectorization (Sentiment)

In [ ]:
def clean_tweet(s):
    s = s.lower()
    s = re.sub(r'http\S+|www\.\S+', ' ', s)      # links: no sentiment information
    s = re.sub(r'@\w+', ' ', s)                  # @airline mentions: identify the airline, not the sentiment
    s = s.replace('#', ' ').replace("'", '')     # keep hashtag words; "don't" -> "dont" so the negation survives tokenising
    s = re.sub(r'[^a-z\s]', ' ', s)              # digits/punctuation/emoji -> space
    return re.sub(r'\s+', ' ', s).strip()
dfB['clean'] = dfB['text'].apply(clean_tweet)
display(dfB[['text', 'clean']].head(5))

In [ ]:
XB_tr_txt, XB_te_txt, yB_tr, yB_te = train_test_split(dfB['clean'], dfB['y'], test_size=TEST_SIZE,
                                                      random_state=RANDOM_STATE, stratify=dfB['y'])
vecB = TfidfVectorizer(ngram_range=(1, 2), min_df=2)    # unigrams + bigrams ("not good"), ignore words seen in <2 train tweets
XB_tr = vecB.fit_transform(XB_tr_txt)                   # fit on TRAIN only
XB_te = vecB.transform(XB_te_txt)
print('Train/test:', XB_tr.shape[0], '/', XB_te.shape[0], '| vocabulary size (features):', XB_tr.shape[1])

### Answers — Part 6 (cleaning decisions beyond the SMS pipeline)
Tweets are messier than SMS messages, so I did four extra things. **(1) Removed URLs**, which carry no sentiment and would just add thousands of unique junk tokens. **(2) Removed @mentions**: nearly every tweet starts with the airline's handle, so the mention identifies *which airline* was addressed rather than *how the customer felt*; keeping it would let the model learn airline identity as a proxy for sentiment. **(3) Stripped `#` but kept the hashtag word** (e.g. `#fail` → `fail`) because hashtags often express the emotion. **(4) Deleted apostrophes** so that "don't" becomes `dont` instead of being split into `don` + a dropped `t` — this keeps negations as usable tokens. I also used **TF-IDF with unigrams + bigrams (`min_df=2`)** so that short phrases such as "not good" or "thank you" can be features; the ablation in Part 7 tests whether that helped. As in Task A, the vectorizer is fit on the training tweets only.

## 7. Task B: Model & Evaluation (Sentiment)

In [ ]:
nbB = MultinomialNB().fit(XB_tr, yB_tr)
predB = nbB.predict(XB_te)
mB = get_metrics(yB_te, predB)
mB['Negative-class recall'] = recall_score(yB_te, predB, pos_label=0)
display(pd.DataFrame([mB], index=['MultinomialNB (positive = 1)']).round(4))
cmB = confusion_matrix(yB_te, predB); tnB, fpB, fnB, tpB = cmB.ravel()
ConfusionMatrixDisplay(cmB, display_labels=['negative', 'positive']).plot(cmap='Blues'); plt.title('Task B — confusion matrix (test)'); plt.show()
baseB = max(yB_te.mean(), 1 - yB_te.mean())
say(f'''**Your numbers:** accuracy **{mB['Accuracy']:.4f}**, precision **{mB['Precision']:.4f}**, recall **{mB['Recall']:.4f}**, F1 **{mB['F1']:.4f}** (positive class);
recall on the negative class = {mB['Negative-class recall']:.4f}. Always predicting "negative" would score {baseB:.3f} accuracy.
Errors: **{fpB}** negative tweets called positive, **{fnB}** positive tweets called negative.''')

### Interpretation of the confusion matrix
Because negative tweets dominate, the model sees many more negative examples and tends to be better at recognising them; the row for *positive* tweets (bottom row) shows how many genuinely positive tweets are mislabeled as negative. Compare the positive-class recall with the negative-class recall in the table.

In [ ]:
def run_b(texts, vec):
    tr, te, ytr, yte = train_test_split(texts, dfB['y'], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=dfB['y'])
    m = MultinomialNB().fit(vec.fit_transform(tr), ytr); return get_metrics(yte, m.predict(vec.transform(te)))
abl = pd.DataFrame({
    'raw text (lowercase only), unigrams':   run_b(dfB['text'].str.lower(), CountVectorizer()),
    'cleaned, unigrams (TF-IDF)':            run_b(dfB['clean'], TfidfVectorizer()),
    'cleaned, uni+bigrams (TF-IDF) — FINAL': run_b(dfB['clean'], TfidfVectorizer(ngram_range=(1, 2), min_df=2))}).T
display(abl.round(4))

# --- does the "naive" independence assumption hurt on negation? ---
neg_pat = r"\b(?:not|no|never|dont|cant|wont|didnt|isnt|cannot|doesnt|wasnt|couldnt)\b"
has_neg = XB_te_txt.str.contains(neg_pat).values
yt, yp = yB_te.values, predB
def sub(mask):
    return (yt[mask] == yp[mask]).mean(), (yt[mask] == 1).mean(), (yp[mask][yt[mask] == 1] == 1).mean() if (yt[mask] == 1).any() else float('nan')
a1, p1, r1 = sub(has_neg); a0, p0, r0 = sub(~has_neg)
neg_tab = pd.DataFrame({'tweets': [has_neg.sum(), (~has_neg).sum()], 'accuracy': [a1, a0], 'share truly positive': [p1, p0],
                        'recall on positives': [r1, r0]}, index=['contain a negation word', 'no negation word'])
display(neg_tab.round(4))

In [ ]:
say(f'''**Your numbers:** Task B accuracy {mB['Accuracy']:.3f}, F1 {mB['F1']:.3f} vs Task A accuracy {mA['Accuracy']:.3f}, F1 {mA['F1']:.3f}.
Tweets that contain a negation word: accuracy {neg_tab.loc['contain a negation word','accuracy']:.3f} vs {neg_tab.loc['no negation word','accuracy']:.3f} for the rest;
bigram feature set vs unigram-only F1: {abl.iloc[2]['F1']:.3f} vs {abl.iloc[1]['F1']:.3f}.''')

### Answers — Part 7 (Task B vs Task A, and the independence assumption)
Compare the numbers above with Task A: Naive Bayes is expected to do **worse on sentiment than on spam**, most visibly in F1 and recall of the minority (positive) class — accuracy can look high mostly because negative tweets make up the large majority. (If your run shows otherwise, say so and explain it.) The reason connects directly to the "naive" assumption. Spam is formulaic: a handful of words ("free", "claim", "txt", a short code) are individually strong evidence, and it is reasonable to treat them as independent clues. Sentiment lives in *how words combine*: negation ("not good", "never late") flips polarity, sarcasm ("great, another 3-hour delay") uses positive words for a negative meaning, and the same word ("delayed", "thanks") changes meaning with context. A bag of independent words cannot represent any of that, so evidence from individual words is summed incorrectly. The negation table above is the specific evidence from my own run, and the ablation shows that adding bigrams partially recovers phrase-level information that the independence assumption otherwise throws away.

## 8. Task B: Kaggle Submission

Same process as Part 5 using **`Kaggle_TaskB_Sentiment_NaiveBayes.ipynb`**: open the Twitter US Airline Sentiment dataset page → **New Notebook** → **File → Import Notebook** → run all → edit the top summary cell → **Save Version → Save & Run All (Public)**.

**Task B Kaggle notebook URL:** `<paste here>`

## 9. Cross-Task Comparison & vs. Logistic Regression

In [ ]:
cmp1 = pd.DataFrame({'Task A — Spam': {k: mA[k] for k in ['Accuracy','Precision','Recall','F1']},
                     'Task B — Sentiment': {k: mB[k] for k in ['Accuracy','Precision','Recall','F1']}}).T
display(cmp1.round(4))

lrA = LogisticRegression(max_iter=1000).fit(XA_tr, yA_tr)
mLR_A = get_metrics(yA_te, lrA.predict(XA_te))
cmp2 = pd.DataFrame({'Naive Bayes (Task A)': {k: mA[k] for k in ['Accuracy','Precision','Recall','F1']},
                     'Logistic Regression (Task A)': mLR_A}).T
display(cmp2.round(4))

# extra: same comparison on Task B features
lrB = LogisticRegression(max_iter=1000).fit(XB_tr, yB_tr)
mLR_B = get_metrics(yB_te, lrB.predict(XB_te))
cmp3 = pd.DataFrame({'Naive Bayes (Task B)': {k: mB[k] for k in ['Accuracy','Precision','Recall','F1']},
                     'Logistic Regression (Task B)': mLR_B}).T
display(cmp3.round(4))

winA = 'Naive Bayes' if mA['F1'] > mLR_A['F1'] else ('Logistic Regression' if mLR_A['F1'] > mA['F1'] else 'neither (tie)')
winB = 'Naive Bayes' if mB['F1'] > mLR_B['F1'] else ('Logistic Regression' if mLR_B['F1'] > mB['F1'] else 'neither (tie)')
say(f'''**Your numbers:** on Task A, F1 NB = {mA['F1']:.3f} vs LR = {mLR_A['F1']:.3f} → winner: **{winA}**.
On Task B, F1 NB = {mB['F1']:.3f} vs LR = {mLR_B['F1']:.3f} → winner: **{winB}**.''')

### Interpretation of the tables
Table 1 puts the two Naive Bayes results side by side (see Part 7 for why Task B is lower). Table 2 is the required Task A comparison with Logistic Regression on the identical bag-of-words matrix; Table 3 repeats it for Task B as a bonus check.

### Answers — Part 9
The winner on each task is printed above. Typically Logistic Regression is as good as or slightly better than Naive Bayes on F1, and the gap is larger on the sentiment task. That matches the independence assumption: Naive Bayes assumes words are conditionally independent given the class, which over-counts correlated evidence (e.g. "free" + "prize" + "claim" are mutually redundant), whereas Logistic Regression learns weights *jointly*, so it can discount redundant words and use the discriminative ones optimally. On the formulaic spam task the assumption is nearly harmless, so NB is almost as good while being much faster to train and needing no tuning — which is why it is still a classic baseline for text.

## 10. Final Reflection & Conclusion

In [ ]:
say(f'''### Draft conclusion (edit it in your own words)
Naive Bayes suited the **{'spam' if mA['F1'] >= mB['F1'] else 'sentiment'} task** better (F1 {mA['F1']:.2f} on spam vs {mB['F1']:.2f} on sentiment); {'spam is formulaic and a few individually strong words carry the signal, which is exactly what an independent-word model can use' if mA['F1'] >= mB['F1'] else 'explain here why the independence assumption was less damaging on tweets in your run'}.
The most predictive spam words ({', '.join(spam_words[:5])}) matched my intuition about calls to action and prizes, and [write what genuinely surprised you — e.g. that chatty words like "{ham_words[0]}" were so predictive of ham, showing the model learns this corpus's style].
On the same bag-of-words features Logistic Regression reached F1 {mLR_A['F1']:.2f} vs Naive Bayes {mA['F1']:.2f} on spam, so {'the simple probabilistic model was competitive' if mA['F1'] >= mLR_A['F1'] - 0.01 else 'learning joint weights gave a measurable gain'}.
For spam, I decided that false positives are costlier, so precision ({mA['Precision']:.2f}, with {fpA} ham messages blocked) matters more than recall ({mA['Recall']:.2f}).
The independence assumption clearly hurt on tweets: accuracy on tweets containing a negation word was {neg_tab.loc['contain a negation word','accuracy']:.2f} vs {neg_tab.loc['no negation word','accuracy']:.2f} without one, and positive-class recall was only {mB['Recall']:.2f}.
Compared with the tabular models of Labs 3–6, Naive Bayes is far faster and needs almost no tuning, but it cannot model feature interactions.
One limitation of my analysis is that I used a single stratified 80/20 split, so small differences between models could be noise.''')

### Kaggle Notebook URLs (must be public and working)
- Task A (Spam): `<paste URL>`
- Task B (Sentiment): `<paste URL>`